# 08 — LLM Interpretation of Forecasting Results

This notebook builds a financial analysis agent step by step — **from scratch**.

We start by showing what a plain LLM *cannot* do, then progressively add tools until
we have an agent capable of interpreting our forecasting model results in real financial context.

---

### What you will learn
| Step | What we do | Why |
|------|-----------|-----|
| 1 | Call a bare LLM | See its limitations (no real-time data) |
| 2 | Add `yfinance` tools | Give the LLM access to live market data |
| 3 | Build an agent | Let the LLM *decide* which tools to call |
| 4 | Interpret model results | Connect MA/ARIMA/XGBoost/LSTM metrics to real context |

**Prerequisites:** run notebooks 03–07 first to generate prediction CSV files. Add your OpenAI API key to `.env`.

## 1. Setup

In [2]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

import yfinance as yf
from dotenv import load_dotenv
from sklearn.metrics import mean_absolute_error, mean_squared_error

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool

load_dotenv('../.env')
print('API key loaded:', bool(os.getenv('OPENAI_API_KEY')))

API key loaded: True


## 2. The Problem: A Plain LLM Has No Real-Time Data

Before building an agent, let's see what happens when we ask a plain LLM about a stock.

A plain LLM only knows what was in its **training data** — it has a knowledge cutoff and
no access to live prices, news, or financial statements.
This is the fundamental limitation we are solving.

In [3]:
# Initialize the LLM
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Ask a question that requires real-time data
response = llm.invoke('What is the current stock price of Apple?')
print(response.content)

I don't have real-time data access to provide current stock prices. To find the latest stock price of Apple (AAPL), please check a financial news website, a stock market app, or a brokerage platform.


**Result:** The LLM admits it cannot answer — it has no access to real-time data.

The same problem applies to our thesis: a plain LLM cannot interpret whether an ARIMA
prediction was good or bad *relative to what the market actually did*.

**Solution: give the LLM tools it can call to fetch live data on demand.**

## 3. Parameters — Set Your Ticker

Change `TICKER` to analyze any stock. The same agent and tools work for any symbol.

In [4]:
TICKER = 'AAPL'

# Paths to prediction CSVs generated by notebooks 03–07
MODELS = {
    'Moving Average': f'../data/results/moving_average_{TICKER}.csv',
    'ARIMA':          f'../data/results/arima_{TICKER}.csv',
    'XGBoost':        f'../data/results/xgboost_{TICKER}.csv',
    'LSTM':           f'../data/results/lstm_{TICKER}.csv',
}

print(f'Ticker: {TICKER}')
print(f'Models: {list(MODELS.keys())}')

Ticker: AAPL
Models: ['Moving Average', 'ARIMA', 'XGBoost', 'LSTM']


## 4. Load Model Results

Load the prediction CSVs from each forecasting model and compute error metrics:

- **MAE** (Mean Absolute Error) — average absolute dollar error
- **RMSE** (Root Mean Squared Error) — penalizes large errors more
- **MAPE** (Mean Absolute Percentage Error) — error as % of actual price

These metrics become the *context* the agent uses to reason about model quality.

In [5]:
results = {}
metrics = []

for model_name, path in MODELS.items():
    df = pd.read_csv(path, index_col='Date', parse_dates=True)
    results[model_name] = df

# Align all models to the same date index (use XGBoost as reference)
common_index = results['XGBoost'].index
for model_name in results:
    results[model_name] = results[model_name].loc[
        results[model_name].index.isin(common_index)
    ]

# Compute metrics for each model
for model_name, df in results.items():
    actuals     = df['Actual'].values
    predictions = df['Predicted'].values
    mae  = mean_absolute_error(actuals, predictions)
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100
    metrics.append({
        'Model':    model_name,
        'MAE':      round(mae,  4),
        'RMSE':     round(rmse, 4),
        'MAPE (%)': round(mape, 4),
    })

metrics_df = pd.DataFrame(metrics).set_index('Model').sort_values('RMSE')
print(f'Model comparison for {TICKER}:')
print(metrics_df.to_string())

Model comparison for AAPL:
                    MAE     RMSE  MAPE (%)
Model                                     
ARIMA            2.6936   4.0512    1.2012
LSTM             6.3052   8.7706    2.5795
Moving Average   9.3565  11.6698    4.0815
XGBoost         16.9743  23.9767    6.7833


## 5. Define Tools

Tools are regular Python functions decorated with `@tool`.
The **docstring** is what the LLM reads to decide *when* to call each tool.
Writing clear docstrings is critical — the agent is only as smart as its tool descriptions.

We define 6 tools:
1. `get_current_price` — live price + 52-week range
2. `get_historical_prices` — recent closing prices
3. `get_news` — latest headlines
4. `get_balance_sheet` — assets, liabilities, equity
5. `get_income_statement` — revenue, profit, net income
6. `get_forecast_comparison` — our MAE/RMSE/MAPE table

In [6]:
@tool
def get_current_price(ticker: str) -> str:
    """Get the current (real-time) stock price for a given ticker symbol.
    Also returns the 52-week high and low."""
    t    = yf.Ticker(ticker)
    info = t.fast_info
    return (
        f"{ticker} current price: ${info.last_price:.2f} | "
        f"52w high: ${info.year_high:.2f} | "
        f"52w low: ${info.year_low:.2f}"
    )


@tool
def get_historical_prices(ticker: str, period: str = '1mo') -> str:
    """Get historical closing prices for a ticker.
    Period options: 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y."""
    df = yf.download(ticker, period=period, auto_adjust=True, progress=False)
    df.columns = df.columns.get_level_values(0)
    close = df['Close'].tail(10)
    return f"Last 10 closing prices for {ticker} ({period}):\n{close.to_string()}"


@tool
def get_news(ticker: str) -> str:
    """Get the latest news headlines for a given ticker symbol."""
    t    = yf.Ticker(ticker)
    news = t.news[:5]  # last 5 articles
    if not news:
        return f"No news found for {ticker}"
    lines = []
    for article in news:
        content = article.get('content', {})
        title   = content.get('title', 'No title')
        summary = content.get('summary', '')
        lines.append(f"- {title}\n  {summary[:200]}")
    return f"Latest news for {ticker}:\n" + "\n".join(lines)


@tool
def get_balance_sheet(ticker: str) -> str:
    """Get the most recent annual balance sheet for a ticker.
    Returns total assets, total liabilities, and stockholders equity."""
    t  = yf.Ticker(ticker)
    bs = t.balance_sheet
    if bs is None or bs.empty:
        return f"No balance sheet data found for {ticker}"
    key_rows  = ['Total Assets', 'Total Liabilities Net Minority Interest', 'Stockholders Equity']
    available = [r for r in key_rows if r in bs.index]
    return f"Balance sheet for {ticker}:\n{bs.loc[available].iloc[:, :2].to_string()}"


@tool
def get_income_statement(ticker: str) -> str:
    """Get the most recent annual income statement for a ticker.
    Returns total revenue, gross profit, and net income."""
    t   = yf.Ticker(ticker)
    is_ = t.income_stmt
    if is_ is None or is_.empty:
        return f"No income statement data found for {ticker}"
    key_rows  = ['Total Revenue', 'Gross Profit', 'Net Income']
    available = [r for r in key_rows if r in is_.index]
    return f"Income statement for {ticker}:\n{is_.loc[available].iloc[:, :2].to_string()}"


@tool
def get_forecast_comparison(ticker: str) -> str:
    """Get the forecasting model comparison results (MAE, RMSE, MAPE) for a ticker.
    Use this to compare Moving Average, ARIMA, XGBoost, and LSTM model performance."""
    return f"Forecasting model comparison for {ticker}:\n{metrics_df.to_string()}"


# All tools in one list — passed to the agent
tools = [
    get_current_price,
    get_historical_prices,
    get_news,
    get_balance_sheet,
    get_income_statement,
    get_forecast_comparison,
]

print(f'Tools registered: {[t.name for t in tools]}')

Tools registered: ['get_current_price', 'get_historical_prices', 'get_news', 'get_balance_sheet', 'get_income_statement', 'get_forecast_comparison']


## 6. Build the Agent

Now we connect the LLM to the tools using `create_agent`.

The agent works in a **loop**:
1. Receives your question
2. Decides which tool(s) to call
3. Calls the tool and gets the result
4. Decides if it needs more tools, or if it has enough to answer
5. Generates the final response

The **system prompt** shapes how the agent reasons and communicates.
Since this is a thesis project, we tell it to act as a financial educator.

In [7]:
SYSTEM_PROMPT = """You are a financial analysis educator helping students 
understand predictive model performance on financial time series.

When answering questions:
1. First gather relevant data using your tools.
2. Explain concepts in simple terms suitable for students with 
   little or no financial background.
3. Always connect model metrics (MAE, RMSE, MAPE) to real market context.
4. When comparing models, explain WHY one outperforms another — 
   not just which has better numbers.
5. Include caveats: these are educational insights, not trading advice.

You have access to: real-time prices, historical data, news, 
financial statements, and model comparison metrics."""

# Build the agent — it now has access to all 6 tools
agent = create_agent(
    llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print('Agent ready.')

Agent ready.


## 7. Ask the Agent

Now we can ask questions that require **real-time data + reasoning**.
The agent decides which tools to call — you don't need to specify.

Change the `question` below and rerun the cell.

In [8]:
# Edit your question here
question = f"Which forecasting model performed best for {TICKER} and why? Also tell me the current price and latest news."

print(f'Question: {question}')
print('Thinking...\n')

# Invoke the agent — input follows the messages format
result = agent.invoke({'messages': [{'role': 'user', 'content': question}]})

# The last message in the response is the agent's final answer
response_text = result['messages'][-1].content

print('=' * 60)
print(response_text)

Question: Which forecasting model performed best for AAPL and why? Also tell me the current price and latest news.
Thinking...

### Forecasting Model Performance for AAPL

For Apple Inc. (AAPL), the performance of different forecasting models is as follows:

| Model            | MAE    | RMSE   | MAPE (%) |
|------------------|--------|--------|----------|
| **ARIMA**        | 2.6936 | 4.0512 | 1.2012   |
| **LSTM**         | 6.3052 | 8.7706 | 2.5795   |
| **Moving Average** | 9.3565 | 11.6698 | 4.0815   |
| **XGBoost**      | 16.9743 | 23.9767 | 6.7833   |

#### Best Performing Model: ARIMA
The ARIMA model outperformed the others in terms of all three metrics: Mean Absolute Error (MAE), Root Mean Square Error (RMSE), and Mean Absolute Percentage Error (MAPE). 

- **MAE** measures the average magnitude of the errors in a set of forecasts, without considering their direction. A lower MAE indicates better accuracy.
- **RMSE** gives a higher weight to larger errors, making it sensitive to

## 8. Thesis-Focused Example Questions

These questions are designed specifically for the Trabajo Terminal —
they connect the model metrics to real financial context.

Copy any of them into the cell above and rerun.

In [9]:
thesis_questions = [
    # Model comparison
    f"Compare all 4 forecasting models for {TICKER}. Which is best for short-term prediction and why?",
    f"Explain in simple terms what MAE, RMSE, and MAPE mean for the {TICKER} model results.",
    f"Why might ARIMA perform differently than LSTM for {TICKER}? Consider the stock's recent trend.",

    # Market context
    f"What is the current price of {TICKER} and how does it compare to its 52-week high?",
    f"Show {TICKER}'s historical prices for the last 3 months. What trend do you observe?",
    f"What are the latest news headlines for {TICKER} and how might they affect forecasting accuracy?",

    # Financial health
    f"Summarize {TICKER}'s financial health based on its balance sheet and income statement.",
    f"Based on {TICKER}'s financials and model performance, which model would you trust most for a 5-day forecast?",
]

print(f'Example questions for {TICKER}:\n')
for i, q in enumerate(thesis_questions, 1):
    print(f'{i}. {q}\n')

Example questions for AAPL:

1. Compare all 4 forecasting models for AAPL. Which is best for short-term prediction and why?

2. Explain in simple terms what MAE, RMSE, and MAPE mean for the AAPL model results.

3. Why might ARIMA perform differently than LSTM for AAPL? Consider the stock's recent trend.

4. What is the current price of AAPL and how does it compare to its 52-week high?

5. Show AAPL's historical prices for the last 3 months. What trend do you observe?

6. What are the latest news headlines for AAPL and how might they affect forecasting accuracy?

7. Summarize AAPL's financial health based on its balance sheet and income statement.

8. Based on AAPL's financials and model performance, which model would you trust most for a 5-day forecast?



## 9. Run All Thesis Questions

Run the agent on every example question and collect the responses.
Useful for generating a full interpretation report for the thesis.

In [10]:
# Set to True to run all questions (uses API credits)
RUN_ALL = True

all_responses = {}

if RUN_ALL:
    for i, q in enumerate(thesis_questions, 1):
        print(f'[{i}/{len(thesis_questions)}] {q[:80]}...')
        result = agent.invoke({'messages': [{'role': 'user', 'content': q}]})
        all_responses[q] = result['messages'][-1].content
        print('Done.\n')
    print(f'Collected {len(all_responses)} responses.')
else:
    print('RUN_ALL is False — set it to True to run all questions.')

[1/8] Compare all 4 forecasting models for AAPL. Which is best for short-term predicti...
Done.

[2/8] Explain in simple terms what MAE, RMSE, and MAPE mean for the AAPL model results...
Done.

[3/8] Why might ARIMA perform differently than LSTM for AAPL? Consider the stock's rec...
Done.

[4/8] What is the current price of AAPL and how does it compare to its 52-week high?...
Done.

[5/8] Show AAPL's historical prices for the last 3 months. What trend do you observe?...
Done.

[6/8] What are the latest news headlines for AAPL and how might they affect forecastin...
Done.

[7/8] Summarize AAPL's financial health based on its balance sheet and income statemen...
Done.

[8/8] Based on AAPL's financials and model performance, which model would you trust mo...
Done.

Collected 8 responses.


## 10. Save Last Response

Save the most recent agent response to a `.txt` file for the thesis report.

In [11]:
os.makedirs('../data/results', exist_ok=True)

output_path = f'../data/results/llm_interpretation_{TICKER}.txt'

with open(output_path, 'w') as f:
    f.write(f'Ticker: {TICKER}\n')
    f.write(f'Date:   {datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    f.write(f'Question: {question}\n')
    f.write('\n' + '=' * 60 + '\n\n')
    f.write(response_text)

    # Also write all responses if collected
    if all_responses:
        f.write('\n\n' + '=' * 60 + '\n')
        f.write('ALL THESIS QUESTIONS\n')
        f.write('=' * 60 + '\n\n')
        for q, r in all_responses.items():
            f.write(f'Q: {q}\n\n{r}\n\n{"—" * 40}\n\n')

print(f'Saved to {output_path}')

Saved to ../data/results/llm_interpretation_AAPL.txt
